In [6]:
import sys
import csv
import os
import json
import torch
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load .env
load_dotenv(find_dotenv())

# -------------------------
# Config
# -------------------------
SEED = int(os.environ.get("SEED", 42))
DATA_DIR = os.environ.get("DATA_DIR", "/workspace/data")

parsed_docs_dir = os.path.join(DATA_DIR, "parsed_docs")
chunked_docs_dir = os.path.join(DATA_DIR, "chunked_docs")
train_csv_path = os.path.join(parsed_docs_dir, "training_final.csv")
test_csv_path = os.path.join(parsed_docs_dir, "test_final.csv")

# -------------------------
# Environment Info
# -------------------------
print(f"SEED: {SEED}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Data dir: {DATA_DIR}")
print(f"Train CSV path: {train_csv_path}")
print(f"Test CSV path: {test_csv_path}")

SEED: 42
CUDA available: True
PyTorch version: 2.2.0
Data dir: /workspace/data
Train CSV path: /workspace/data/parsed_docs/training_final.csv
Test CSV path: /workspace/data/parsed_docs/test_final.csv


## Chunking Experimental Setup

| Strategy          | Description                         | Typical Chunk Size |
| ----------------- | ----------------------------------- | ------------------ |
| **Section-aware** | Title + abstract sections preserved | 400–600            |
| **Fixed-length**  | Split long text uniformly           | 500                |
| **Short chunks**  | Fine-grained retrieval              | 120–200            |


In [1]:
!pip install transformers accelerate torch

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    return_full_text=False
)

/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [2]:
def summarize_chunk(title, section, chunk):

    prompt = f"""
You are assisting a biomedical retrieval system.

Summarize the key biomedical context of the following text in ONE sentence.

Title: {title}
Section: {section}

Text:
{chunk}

Summary:
"""

    result = llm(prompt)[0]["generated_text"].strip()

    summary = result.split("Summary:")[-1].strip()

    return summary

In [3]:
import pandas as pd
import json
import re
import time
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

start_time = time.time()

df = pd.read_csv("pubmed_articles.csv")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=[
        "\n\n",
        "\n",
        ". ",
    ]
)

chunks = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing articles"):

    pmid = row["pmid"]
    title = str(row["title"]).strip()

    # ---------- TITLE CHUNK ----------
    chunks.append({
        "pmid": pmid,
        "section": "TITLE",
        "chunk_id": f"{pmid}_TITLE",
        "chunk_text": title
    })

    sections = json.loads(row["abstract_sections"])

    for sec in sections:

        label = sec.get("label", "ABSTRACT")
        text = sec.get("text", "")

        text = re.sub(r"<.*?>", "", text).strip()

        # summarize once per section
        section_summary = summarize_chunk(title, label, text)

        # ---------- SHORT SECTION ----------
        if len(text) <= 500:

            contextual_text = (
                f"Context summary: {section_summary}\n\n"
                f"{text}"
            )

            chunks.append({
                "pmid": pmid,
                "section": label,
                "chunk_id": f"{pmid}_{label}_0",
                "chunk_text": contextual_text
            })

        # ---------- LONG SECTION ----------
        else:

            split_texts = splitter.split_text(text)

            for i, chunk in enumerate(split_texts):

                contextual_text = (
                    f"Context summary: {section_summary}\n\n"
                    f"{chunk}"
                )

                chunks.append({
                    "pmid": pmid,
                    "section": label,
                    "chunk_id": f"{pmid}_{label}_{i}",
                    "chunk_text": contextual_text
                })

chunk_df = pd.DataFrame(chunks)

chunk_df.to_csv("chunks_section_contextual.csv", index=False)

end_time = time.time()

print("\nTotal chunks:", len(chunk_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(chunk_df.head())

Processing articles: 100% 1/1 [00:11<00:00, 11.20s/it]


Total chunks: 9
Runtime: 11.23 seconds

Sample output:
       pmid     section               chunk_id  \
0  30415629       TITLE         30415629_TITLE   
1  30415629  BACKGROUND  30415629_BACKGROUND_0   
2  30415629     METHODS     30415629_METHODS_0   
3  30415629     METHODS     30415629_METHODS_1   
4  30415629     RESULTS     30415629_RESULTS_0   

                                          chunk_text  
0  Vitamin D Supplements and Prevention of Cancer...  
1  Context summary: The current research on vitam...  
2  Context summary: The study compares the effect...  
3  Context summary: The study compares the effect...  
4  Context summary: Vitamin D supplementation did...  


### Strategy 1 — Section-Aware Chunking

In [8]:
import pandas as pd
import json
import re
import html
import unicodedata
from tqdm import tqdm
from pathlib import Path

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

start_time = time.time()

# =========================
# Load Data
# =========================

df = pd.read_csv(train_csv_path)


# =========================
# Tokenizer (for token chunking)
# =========================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)


splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=400,
    chunk_overlap=60,
    separators=[
        "\n\n",
        "\n",
        ". ",
    ]
)


# =========================
# PubMed Text Cleaning
# =========================

def clean_pubmed_text(text):

    if pd.isna(text):
        return ""

    # remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # decode HTML entities
    text = html.unescape(text)

    # normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # remove strange characters
    text = text.encode("ascii", "ignore").decode()

    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =========================
# Chunking
# =========================

chunks = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    pmid = str(row["pmid"])
    title = clean_pubmed_text(row.get("title", ""))
    text = clean_pubmed_text(row.get("text", ""))

    topic_id = row.get("topic_id")
    category = row.get("category")

    # =========================
    # TITLE CHUNK
    # =========================

    if title:
        chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "TITLE",
            "chunk_id": f"{pmid}_TITLE",
            "title": title,
            "chunk_text": title
        })


    # =========================
    # ABSTRACT / TEXT SECTION
    # =========================

    if not text:
        continue

    token_len = len(tokenizer.encode(text))


    # if short → keep whole
    if token_len <= 400:

        chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "ABSTRACT",
            "chunk_id": f"{pmid}_ABSTRACT_0",
            "title": title,
            "chunk_text": text
        })

    else:

        split_texts = splitter.split_text(text)

        for i, chunk in enumerate(split_texts):

            chunks.append({
                "pmid": pmid,
                "topic_id": topic_id,
                "category": category,
                "section": "ABSTRACT",
                "chunk_id": f"{pmid}_ABSTRACT_{i}",
                "title": title,
                "chunk_text": chunk
            })


# =========================
# Save
# =========================

chunk_df = pd.DataFrame(chunks)

output_dir = Path(chunked_docs_dir)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks_section_400token.csv"
chunk_df.to_csv(output_path, index=False)

end_time = time.time()

print("\nTotal chunks:", len(chunk_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(chunk_df.head())

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
100% 39442/39442 [00:55<00:00, 713.14it/s]



Total chunks: 87996
Runtime: 58.59 seconds

Sample output:
       pmid  topic_id       category   section             chunk_id  \
0  15829955         8  Rare Diseases     TITLE       15829955_TITLE   
1  15829955         8  Rare Diseases  ABSTRACT  15829955_ABSTRACT_0   
2  15617541         8  Rare Diseases     TITLE       15617541_TITLE   
3  15617541         8  Rare Diseases  ABSTRACT  15617541_ABSTRACT_0   
4  12239580         8  Rare Diseases     TITLE       12239580_TITLE   

                                               title  \
0  A common sex-dependent mutation in a RET enhan...   
1  A common sex-dependent mutation in a RET enhan...   
2  Studying the genetics of Hirschsprung's diseas...   
3  Studying the genetics of Hirschsprung's diseas...   
4  Hirschsprung, RET-SOX and beyond: the challeng...   

                                          chunk_text  
0  A common sex-dependent mutation in a RET enhan...  
1  The identification of common variants that con...  
2  Studying

In [9]:
# character length
chunk_df["char_len"] = chunk_df["chunk_text"].str.len()

# word count
chunk_df["word_count"] =chunk_df["chunk_text"].str.split().str.len()

print(chunk_df.head())

       pmid     section               chunk_id  \
0  30415629       TITLE         30415629_TITLE   
1  30415629  BACKGROUND  30415629_BACKGROUND_0   
2  30415629     METHODS     30415629_METHODS_0   
3  30415629     METHODS     30415629_METHODS_1   
4  30415629     RESULTS     30415629_RESULTS_0   

                                          chunk_text  char_len  word_count  
0  Vitamin D Supplements and Prevention of Cancer...        74          10  
1  It is unclear whether supplementation with vit...       151          23  
2  We conducted a nationwide, randomized, placebo...       382          68  
3  . Primary end points were invasive cancer of a...       358          51  
4  A total of 25,871 participants, including 5106...       408          65  


### Strategy 2 — Fixed Length Chunk (>500)

In [4]:
import pandas as pd
import json
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

df = pd.read_csv("pubmed_articles.csv")

splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=[""]
)

fixed_chunks = []

for _, row in df.iterrows():

    pmid = row["pmid"]
    title = row["title"]

    # title chunk
    fixed_chunks.append({
        "pmid": pmid,
        "chunk_id": f"{pmid}_TITLE",
        "chunk_text": title
    })

    sections = json.loads(row["abstract_sections"])

    # join all sections
    full_text = " ".join(
        re.sub(r"<.*?>", "", sec.get("text", "")).strip()
        for sec in sections
    )

    split_texts = splitter_500.split_text(full_text)

    for i, chunk in enumerate(split_texts):

        fixed_chunks.append({
            "pmid": pmid,
            "chunk_id": f"{pmid}_F500_{i}",
            "chunk_text": chunk
        })

fixed_df = pd.DataFrame(fixed_chunks)

fixed_df.to_csv("chunks_fixed500.csv", index=False)

In [5]:
# character length
fixed_df["char_len"] = fixed_df["chunk_text"].str.len()

# word count
fixed_df["word_count"] = fixed_df["chunk_text"].str.split().str.len()

print(fixed_df.head())

       pmid         chunk_id  \
0  30415629   30415629_TITLE   
1  30415629  30415629_F500_0   
2  30415629  30415629_F500_1   
3  30415629  30415629_F500_2   
4  30415629  30415629_F500_3   

                                          chunk_text  char_len  word_count  
0  Vitamin D Supplements and Prevention of Cancer...        74          10  
1  It is unclear whether supplementation with vit...       500          84  
2  cardiovascular disease among men 50 years of a...       499          77  
3  results of the comparison of vitamin D with pl...       500          79  
4  ence interval [CI], 0.88 to 1.06; P=0.47). A m...       500          88  


### Strategy 3 — Very Short Chunks

In [6]:
splitter_150 = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=[""]
)

short_chunks = []

for _, row in df.iterrows():

    pmid = row["pmid"]
    title = row["title"]

    # title chunk
    short_chunks.append({
        "pmid": pmid,
        "chunk_id": f"{pmid}_TITLE",
        "chunk_text": title
    })

    sections = json.loads(row["abstract_sections"])

    full_text = " ".join(
        re.sub(r"<.*?>", "", sec.get("text", "")).strip()
        for sec in sections
    )

    split_texts = splitter_150.split_text(full_text)

    for i, chunk in enumerate(split_texts):

        short_chunks.append({
            "pmid": pmid,
            "chunk_id": f"{pmid}_S150_{i}",
            "chunk_text": chunk
        })

short_df = pd.DataFrame(short_chunks)

short_df.to_csv("chunks_short150.csv", index=False)

In [7]:
# character length
short_df["char_len"] = short_df["chunk_text"].str.len()

# word count
short_df["word_count"] = short_df["chunk_text"].str.split().str.len()

print(short_df.head())

       pmid         chunk_id  \
0  30415629   30415629_TITLE   
1  30415629  30415629_S150_0   
2  30415629  30415629_S150_1   
3  30415629  30415629_S150_2   
4  30415629  30415629_S150_3   

                                          chunk_text  char_len  word_count  
0  Vitamin D Supplements and Prevention of Cancer...        74          10  
1  It is unclear whether supplementation with vit...       150          23  
2  randomized trials are limited. We conducted a ...       149          20  
3  rial design, of vitamin D3 (cholecalciferol) a...       150          31  
4  ds at a dose of 1 g per day for the prevention...       150          33  


## Load the CSV and parse the abstracts column

In [20]:
# Read only first 10 rows
df_train_raw = pd.read_csv(train_csv_path, engine="python")
df_train = df_train_raw.copy()

In [21]:
print("Rows:", len(df_train))
print("Columns:", df_train.columns.tolist())

df_train.info()

Rows: 39418
Columns: ['question', 'type', 'text', 'category', 'topic_id', 'confidence', 'pmid', 'source_url', 'n_sources']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39418 entries, 0 to 39417
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   question    39418 non-null  object 
 1   type        39418 non-null  object 
 2   text        39417 non-null  object 
 3   category    39418 non-null  object 
 4   topic_id    39418 non-null  int64  
 5   confidence  39418 non-null  float64
 6   pmid        39418 non-null  int64  
 7   source_url  39418 non-null  object 
 8   n_sources   39418 non-null  int64  
dtypes: float64(1), int64(3), object(5)
memory usage: 2.7+ MB


In [26]:
df_train.isnull().sum()

question      0
type          0
text          1
category      0
topic_id      0
confidence    0
pmid          0
source_url    0
n_sources     0
dtype: int64

In [23]:
df_train["question"].value_counts()

question
Which is the causative agent of malaria?                                                                                                    105
In which proteins is the chromodomain present?                                                                                              102
What tyrosine kinase, involved in a Philadelphia- chromosome positive chronic myelogenous leukemia, is the target of Imatinib (Gleevec)?     79
Is miR-21 related to carcinogenesis?                                                                                                         69
What clinical conditions influence the prognostic after the liver metastasis resection from colorectal cancer patients?                      65
                                                                                                                                           ... 
What methodology does the HercepTest use?                                                                                      

In [27]:
df_train["category"].value_counts()

category
Other                    5690
Pharmacology & Drugs     5511
Molecular Biology        4503
Rare Diseases            3909
Cancer & Oncology        3587
Genetics & Mutations     3536
Infectious Disease       3145
Immunology               3036
Cardiology & Heart       2150
Surgery & Procedures     1632
Neurology & Brain        1345
Mental Health             445
Diagnostics & Imaging     393
Metabolism & Diabetes     353
Clinical Guidelines       183
Name: count, dtype: int64

In [28]:
import matplotlib.pyplot as plt

df["category"].value_counts().plot(kind="bar")
plt.title("Category Distribution")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [6]:
# Extract abstract dataframe
df_abstracts = df[["pmid", "text"]].copy()

# Drop duplicate PMIDs
df_abstracts = df_abstracts.drop_duplicates(subset=["pmid"]).reset_index(drop=True)

print("Rows after dedup:", len(df_abstracts))
print()

# Display nicely
for i, row in df_abstracts.head(5).iterrows():
    print("="*80)
    print(f"PMID: {row['pmid']}")
    print("-"*80)
    print(row["text"])
    print(len(row["text"]))
    print()

Rows after dedup: 95

PMID: 15829955
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    OBJECTIVE: To analyze the prognostic factors o...
Name: 0, dtype: object
2

PMID: 15617541
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Neuromyelitis Optica Spectrum Diso...
Name: 1, dtype: object
2

PMID: 12239580
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Severe forms of primary dystonia a...
Name: 2, dtype: object
2

PMID: 6650562
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Recent studies linking radiation e...
Name: 3, dtype: object
2

PMID: 20598273
------------

In [6]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.embedding_manager import MedicalEmbeddingManager

embedding_manager = MedicalEmbeddingManager()

query_vec = embedding_manager.embed_query(
    "What causes Alzheimer's disease?"
)

chunks = [
    "Amyloid beta accumulation is a key factor in Alzheimer's disease.",
    "Neuroinflammation contributes to neurodegenerative disorders.",
    "Tau protein aggregation leads to neuronal damage."
]

doc_vecs = embedding_manager.embed_documents(chunks)

print("Query embedding dimension:", len(query_vec))
print("First 10 values of query embedding:", query_vec[:10])

print("\nNumber of document embeddings:", len(doc_vecs))
print("Document embedding dimension:", len(doc_vecs[0]))
print("First 10 values of first document embedding:", doc_vecs[0][:10])

/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Query embedding dimension: 768
First 10 values of query embedding: [-0.02226661890745163, -0.04278154671192169, -0.0294787660241127, -0.02906412072479725, 0.004392766393721104, 0.004281898029148579, -0.01897631213068962, 0.026256967335939407, 0.014577195979654789, 0.0341288186609745]

Number of document embeddings: 3
Document embedding dimension: 768
First 10 values of first document embedding: [-0.031111201271414757, -0.03551964461803436, -0.03676166757941246, -0.04713869467377663, -0.008105887100100517, -0.00807227473706007, -0.030369000509381294, 0.03008858673274517, 0.017857875674962997, 0.027921907603740692]


In [9]:
import torch
import transformers
import sentence_transformers

print(torch.__version__)
print(transformers.__version__)
print(sentence_transformers.__version__)
print(embedding_manager.embedding_model)

2.2.0
4.41.2
2.7.0
pritamdeka/S-PubMedBert-MS-MARCO
